In [5]:
import numpy as np
from datetime import datetime, timedelta

# 1. THE CHAT PARSER

def parse_chat(file_path):
    parsed_messages = []
    stats = {
        'system': 0,
        'media': 0,
        'deleted': 0,
        'real': 0
    }

    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    for line in lines:
        line = line.strip()
        if not line:
            continue

        if ' - ' not in line[:25]:
            continue

        ts_str, rest = line.split(' - ', 1)

        if ': ' not in rest:
            stats['system'] += 1
            continue

        sender, msg_text = rest.split(': ', 1)

        try:
            ts = datetime.strptime(ts_str, "%d/%m/%y, %H:%M")
        except ValueError:
            continue

        is_media = False
        is_deleted = False

        if msg_text == '<Media omitted>':
            stats['media'] += 1
            is_media = True
        elif msg_text == 'This message was deleted':
            stats['deleted'] += 1
            is_deleted = True
        else:
            stats['real'] += 1

        parsed_messages.append({
            'timestamp': ts,
            'sender': sender,
            'text': msg_text,
            'is_media': is_media,
            'is_deleted': is_deleted
        })

    return parsed_messages, stats

messages, parse_stats = parse_chat('hostel_bois.txt')


# 2. GROUP OVERVIEW

participants = set()
person_counts = {}

min_date = messages[0]['timestamp']
max_date = messages[0]['timestamp']

for msg in messages:
    sender = msg['sender']
    ts = msg['timestamp']

    participants.add(sender)
    person_counts[sender] = person_counts.get(sender, 0) + 1

    if ts < min_date: min_date = ts
    if ts > max_date: max_date = ts

total_messages = len(messages)
total_days = (max_date - min_date).days + 1
sorted_participants = sorted(person_counts.items(), key=lambda x: x[1], reverse=True)


# 3. MOST ACTIVE DAY AND HOUR

day_counts = {}
hour_counts = {}

for msg in messages:
    day_str = msg['timestamp'].strftime("%d %B %Y")
    hour = msg['timestamp'].hour

    day_counts[day_str] = day_counts.get(day_str, 0) + 1
    hour_counts[hour] = hour_counts.get(hour, 0) + 1

busiest_day = max(day_counts.items(), key=lambda x: x[1])
busiest_hour = max(hour_counts.items(), key=lambda x: x[1])


# 4. ACTIVITY HEATMAP (NUMPY)

p_list = [p[0] for p in sorted_participants]
p_indices = {name: idx for idx, name in enumerate(p_list)}

heatmap_matrix = np.zeros((len(p_list), 24), dtype=int)

for msg in messages:
    row = p_indices[msg['sender']]
    col = msg['timestamp'].hour
    heatmap_matrix[row, col] += 1

def get_block_char(val, max_val):
    if max_val == 0 or val == 0: return '.'
    ratio = val / max_val
    if ratio <= 0.25: return '░'
    elif ratio <= 0.50: return '▒'
    elif ratio <= 0.75: return '▓'
    else: return '█'


# 5. TOP WORDS

word_counts = {}
stop_words = {'i', 'is', 'the', 'a', 'and', 'or', 'to', 'of', 'in', 'on', 'for', 'it', 'this', 'that', 'with', 'my', 'me', 'am', 'was', 'so', 'you', 'are'}

for msg in messages:
    if msg['is_media'] or msg['is_deleted']:
        continue

    clean_text = ""
    for char in msg['text'].lower():
        if char.isalpha() or char.isspace():
            clean_text += char

    words = clean_text.split()
    for w in words:
        if w not in stop_words and len(w) > 2:
            word_counts[w] = word_counts.get(w, 0) + 1

top_words = sorted(word_counts.items(), key=lambda x: x[1], reverse=True)[:5]


# 6. RESPONSE SPEED & SILENT STREAKS

response_gaps = {p: [] for p in p_list}
active_dates = {p: set() for p in p_list}

last_sender = None
last_ts = None

for msg in messages:
    sender = msg['sender']
    ts = msg['timestamp']
    date_str = ts.strftime("%Y-%m-%d")

    active_dates[sender].add(date_str)

    if last_sender and last_sender != sender:
        gap = (ts - last_ts).total_seconds()

        if gap < 86400:
            response_gaps[sender].append(gap)

    last_sender = sender
    last_ts = ts

avg_response = {}
for p in p_list:
    if response_gaps[p]:
        avg_response[p] = sum(response_gaps[p]) / len(response_gaps[p])
    else:
        avg_response[p] = 0

longest_streaks = {p: 0 for p in p_list}
for p in p_list:
    current_streak = 0
    max_streak = 0
    curr_date = min_date
    while curr_date <= max_date:
        if curr_date.strftime("%Y-%m-%d") not in active_dates[p]:
            current_streak += 1
            if current_streak > max_streak:
                max_streak = current_streak
        else:
            current_streak = 0
        curr_date += timedelta(days=1)
    longest_streaks[p] = max_streak

# 7. PERSONALITY ARCHETYPES

archetypes = {}
caring_keywords = ['okay', 'safe', 'eat', 'sleep', 'take care', 'are you', 'please', 'reminder', 'drink water', "don't forget"]

for p in p_list:
    p_msgs = [m for m in messages if m['sender'] == p and not m['is_media'] and not m['is_deleted']]
    total_p_msgs = len(p_msgs)

    if total_p_msgs == 0:
        continue

    scores = {}


    bursts = []
    curr_burst = 0
    prev_sender = None
    for m in messages:
        if m['sender'] == p:
            if prev_sender == p:
                curr_burst += 1
            else:
                curr_burst = 1
        else:
            if prev_sender == p:
                bursts.append(curr_burst)
                curr_burst = 0
        prev_sender = m['sender']

    avg_burst = sum(bursts)/len(bursts) if bursts else 0
    scores['THE SPAMMER'] = avg_burst if avg_burst > 3 else 0


    mom_score = 0
    for m in p_msgs:
        txt = m['text'].lower()
        for kw in caring_keywords:
            if kw in txt:
                mom_score += 1
    scores['THE GROUP MOM'] = mom_score


    night_count = sum(1 for m in p_msgs if m['timestamp'].hour >= 23 or m['timestamp'].hour < 5)
    night_pct = night_count / total_p_msgs
    scores['THE NIGHT OWL'] = night_pct if night_pct > 0.6 else 0

    avg_words = sum(len(m['text'].split()) for m in p_msgs) / total_p_msgs
    scores['THE STORYTELLER'] = avg_words if avg_words > 30 else 0

    drama_count = sum(1 for m in p_msgs if (m['text'].isupper() and len(m['text']) > 2) or m['text'].count('!') >= 2)
    drama_pct = drama_count / total_p_msgs
    scores['THE DRAMA QUEEN'] = drama_pct if drama_pct > 0.3 else 0

    ghost_pct = longest_streaks[p] / total_days
    scores['THE GHOST'] = ghost_pct if ghost_pct > 0.6 else 0

    best_archetype = max(scores.items(), key=lambda x: x[1])
    if best_archetype[1] > 0:
        archetypes[p] = best_archetype
    else:
        archetypes[p] = ('THE OBSERVER', 0)


# 8. FINAL REPORT GENERATION

print("========================================================")
print(f"{'GroupDNA REPORT':^56}")
print("========================================================")
print(f"Group          : Hostel Bois 4ever")
print(f"Total messages : {total_messages:,}")
print(f"Participants   : {len(p_list)}")
print(f"Period         : {min_date.strftime('%d %B %Y')} to {max_date.strftime('%d %B %Y')} ({total_days} days)")
print(f"Busiest day    : {busiest_day[0]} ({busiest_day[1]} messages)")
print(f"Busiest hour   : {busiest_hour[0]:02d}:00 - {busiest_hour[0]+1:02d}:00")
print("========================================================")

print("\nMESSAGES PER PERSON")
for name, count in sorted_participants:
    pct = (count / total_messages) * 100
    print(f"  {name:<10} : {count:>4} ({pct:>4.1f}%)")

print("\nACTIVITY HEATMAP (messages by hour)")
print("           00 03 06 09 12 15 18 21")
for name in p_list:
    row_idx = p_indices[name]
    max_val = heatmap_matrix[row_idx].max()
    row_viz = ""
    for h in [0, 3, 6, 9, 12, 15, 18, 21]:
        val = heatmap_matrix[row_idx, h:h+3].sum()
        row_viz += f"{get_block_char(val, max_val * 3)}  "
    print(f"  {name:<10} {row_viz}")

print("\nTHIS GROUP'S FAVOURITE WORDS")
for word, count in top_words:
    bar = '█' * (count // 20)
    print(f"  {word:<10} {count:>4} {bar}")

print("\nRESPONSE PATTERNS")
fastest = min(avg_response.items(), key=lambda x: x[1] if x[1] > 0 else float('inf'))
slowest = max(avg_response.items(), key=lambda x: x[1])
print(f"  Fastest replier: {fastest[0]} (avg {fastest[1]/60:.1f} minutes)")
print(f"  Slowest replier: {slowest[0]} (avg {slowest[1]/3600:.1f} hours)")

print("\nLONGEST SILENT STREAKS")
for name, streak in sorted(longest_streaks.items(), key=lambda x: x[1], reverse=True):
    print(f"  {name:<10} : {streak} days")

print("\nPERSONALITY ARCHETYPES")
for name, (arch, score) in archetypes.items():
    print(f"  {name:<10} -> {arch:<18}")

print("========================================================")
print(f"{'Generated by GroupDNA':^56}")
print(f"{'Built with Python + NumPy':^56}")
print("========================================================")

                    GroupDNA REPORT                     
Group          : Hostel Bois 4ever
Total messages : 3,174
Participants   : 6
Period         : 01 April 2024 to 30 May 2024 (60 days)
Busiest day    : 04 May 2024 (76 messages)
Busiest hour   : 18:00 - 19:00

MESSAGES PER PERSON
  Rahul      :  953 (30.0%)
  Priya      :  718 (22.6%)
  Neha       :  635 (20.0%)
  Aman       :  490 (15.4%)
  Karan      :  354 (11.2%)
  Vikas      :   24 ( 0.8%)

ACTIVITY HEATMAP (messages by hour)
           00 03 06 09 12 15 18 21
  Rahul      ░  ░  ░  ░  ▒  ▓  ▓  ▓  
  Priya      .  .  ▒  █  █  ▓  ▓  ▒  
  Neha       .  ░  ▒  ▓  ▓  ▓  █  ▒  
  Aman       ▓  ▓  .  .  ░  ░  ░  ▒  
  Karan      .  .  ░  ▒  █  ▓  ▓  ▒  
  Vikas      .  .  ▒  ░  ▒  ▓  ▓  ▒  

THIS GROUP'S FAVOURITE WORDS
  how         321 ████████████████
  guys        318 ███████████████
  about       274 █████████████
  hai         268 █████████████
  today       257 ████████████

RESPONSE PATTERNS
  Fastest replier: Rahul (avg 34.9